# Evaluate the paper weights only

This notebook does not train. It loads the result_4 checkpoints and scores the same patient-disjoint validation and test images.

Upload `notebooks/res4/weights/` as a Kaggle dataset (three files: `concept_warmup_resnet18.pt`, `deepproblog_model.pth`, `test_prob.npy`) and run this notebook with the same CBIS-DDSM input used for result_4. CPU is enough. Train images are not loaded.

The run stops if the Stage 2 weights do not reproduce the paper's validation concept check (90.5 / 72.3 / 70.1 / 82.0) or the saved `test_prob.npy`.


## 1) Install packages

**What:** install DeepProbLog, scikit-learn, pandas, and Pillow in this kernel.

**Why:** Kaggle runtimes do not always include DeepProbLog. This cell is safe to re-run.


In [ ]:
%pip install -q deepproblog scikit-learn pandas pillow


## 2) Imports

**What:** load libraries for tables, images, PyTorch, and plots.

**Why:** later cells need pandas for CSVs, matplotlib to show mammograms, and torch for the concept networks.


In [ ]:
import json
import os
import random
from datetime import datetime, timezone
from itertools import product
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from torchvision import models, transforms
from torchvision.models import ResNet18_Weights

print("pandas:", pd.__version__)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


## 3) Config

**What:** one settings cell for the whole notebook.

**Why:** change smoke-test size, training length, augmentation, and the hyperparameter grid here.

- `SMOKE_TEST=True` — tiny wiring test (few patients, 1 epoch).
- `SMOKE_TEST=False` — full split and 15 epochs.
- `N_AUGMENT` — extra random views per **training** image only. Validation and test stay clean.
- `BACKBONE` — `resnet18` (ImageNet transfer) or `tiny` (from-scratch CNN for smoke tests).
- `CONCEPT_WARMUP_EPOCHS` — supervised CE on CSV concepts before DeepProbLog (0 = skip).
- `FREEZE_BACKBONE_EPOCHS` — train concept heads only, then unfreeze the ResNet trunk at a lower LR.
- `RUN_NEURAL_BASELINE` — train a pure ResNet18 malignant/benign classifier on the same splits (no DeepProbLog).
- `PATCH_MODE` — `resize` / `center` / `random`: how the ROI crop becomes a fixed `IMAGE_SIZE` patch (see section 8).
- `BALANCE_CLASSES=True` — malignant training rows get extra augmented copies until both classes are equal in size.
- `THRESHOLD_MODE` / `TARGET_SENSITIVITY` — how the decision threshold on `P(malignant)` is chosen on validation. `0.5` under-calls malignant; `target_sensitivity` lowers it until validation recall reaches the target.
- `RUN_GRID_SEARCH=True` — try each `(lr, batch_size)` for `SEARCH_EPOCHS`, pick the best validation macro F1, then retrain with `EPOCHS`.
- `USE_SYNTHETIC_IF_MISSING=False` — missing JPEGs must raise an error.
- `FORCE_CPU=False` — use T4 if the CUDA probe succeeds.


In [ ]:
# ===================== ONE CONFIG FOR EVERYTHING =====================
SMOKE_TEST = False                 # True = tiny wiring test; False = full run
SMOKE_N_PATIENTS = 20             # patients per official split (data prep)
SMOKE_N_SAMPLES = 24              # max rows used for DeepProbLog tensors
VAL_SIZE = 0.15
RANDOM_SEED = 42

EPOCHS = 1 if SMOKE_TEST else 15         # full run: 15 epochs
BATCH_SIZE = 1 if SMOKE_TEST else 2      # default if grid search is off; drop to 2 on OOM
# Medical mammography: higher res keeps fine margin / spiculation detail.
# 224 is the next step (ImageNet-friendly); 256 is ok on T4 with batch=2.
# 512+ is usually too slow for DeepProbLog ExactEngine with 4 nets.
IMAGE_SIZE = 64 if SMOKE_TEST else 224
# How to turn a ROI crop into a fixed tensor (one patch per image — DeepProbLog-friendly):
#   "resize" = stretch whole crop to IMAGE_SIZE (old behaviour; can smear margins)
#   "center" = scale so the short side >= IMAGE_SIZE, then center-crop (val/test default)
#   "random" = RandomResizedCrop on train; center-crop on val/test (recommended)
PATCH_MODE = "resize" if SMOKE_TEST else "random"
# Perception backbone for the four DeepProbLog concept predicates.
#   "resnet18" — ImageNet-pretrained ResNet18 shared trunk + 4 heads (recommended)
#   "tiny"     — from-scratch 3-layer CNN (fast smoke / ablation)
BACKBONE = "tiny" if SMOKE_TEST else "resnet18"
FREEZE_BACKBONE_EPOCHS = 0 if SMOKE_TEST else 2   # DPL phase: unused if concept warm-up ran
BACKBONE_LR_MULT = 0.1                           # trunk LR = LR * this after unfreeze
# Supervised concept warm-up (CE on kind/shape/margin/calc) before DeepProbLog.
CONCEPT_WARMUP_EPOCHS = 1 if SMOKE_TEST else 8
CONCEPT_WARMUP_FREEZE_EPOCHS = 0 if SMOKE_TEST else 2  # freeze trunk for first N warm-up epochs
CONCEPT_WARMUP_BATCH = 4 if SMOKE_TEST else 16
RUN_NEURAL_BASELINE = not SMOKE_TEST             # pure ResNet18 mal/ben on same splits
BASELINE_EPOCHS = 1 if SMOKE_TEST else 15
LR = 0.0003                                # default if grid search is off
USE_SYNTHETIC_IF_MISSING = False  # missing JPEG must fail; do not invent images
FORCE_CPU = False                 # use GPU if the probe succeeds (T4). P100 still falls back to CPU.
SHOW_N_IMAGES = 8                 # pictures to draw in each image grid

# Train-only augmentation: each original train image is kept, plus N_AUGMENT random copies.
N_AUGMENT = 1 if SMOKE_TEST else 2

# Class balance. Malignant is ~41% of train, so at 0.5 the model under-calls malignant.
# With BALANCE_CLASSES=True the malignant rows get extra augmented copies until both
# classes have the same number of training images. Validation and test are untouched.
BALANCE_CLASSES = True

# Decision threshold on P(malignant). The threshold is chosen on VALIDATION only and
# then applied unchanged to TEST, so the test numbers stay honest.
#   "fixed"              -> use DECISION_THRESHOLD as is
#   "target_sensitivity" -> lowest threshold whose validation sensitivity >= TARGET_SENSITIVITY
#   "youden"             -> maximize sensitivity + specificity - 1
#   "f1_macro"           -> maximize macro F1
THRESHOLD_MODE = "target_sensitivity"
TARGET_SENSITIVITY = 0.80
DECISION_THRESHOLD = 0.5

# Small grid on validation macro F1. Keep it small: DeepProbLog exact inference is slow.
RUN_GRID_SEARCH = True
SEARCH_EPOCHS = 1 if SMOKE_TEST else 3
GRID_LEARNING_RATES = [3e-4, 1e-3] if SMOKE_TEST else [3e-4, 1e-3, 3e-3]
GRID_BATCH_SIZES = [2] if SMOKE_TEST else [2, 4]

DATASET_HANDLE = "awsaf49/cbis-ddsm-breast-cancer-image-dataset"
KAGGLE_INPUT = Path("/kaggle/input/cbis-ddsm-breast-cancer-image-dataset")
KAGGLE_WORKING = Path("/kaggle/working")
LOCAL_DATA = Path("../data")

CSV_FILENAMES = {
    "mass_train": "mass_case_description_train_set.csv",
    "mass_test": "mass_case_description_test_set.csv",
    "calc_train": "calc_case_description_train_set.csv",
    "calc_test": "calc_case_description_test_set.csv",
}

PROGRAM_FALLBACK = """
nn(kind_net, [X], Y, [mass, calcification]) :: lesion_kind(X, Y).
nn(shape_net, [X], Y, [regular, irregular, unknown]) :: shape(X, Y).
nn(margin_net, [X], Y, [circumscribed, spiculated, other, unknown]) :: margin(X, Y).
nn(calc_net, [X], Y, [absent, typically_benign, suspicious]) :: calc(X, Y).

malignant(X) :- lesion_kind(X, mass), shape(X, irregular).
malignant(X) :- lesion_kind(X, mass), margin(X, spiculated).
malignant(X) :- lesion_kind(X, calcification), calc(X, suspicious).
""".strip()

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def pick_device(force_cpu=False):
    """Prefer CUDA, but fall back to CPU if the binary can't run on this GPU.
    Common on Kaggle when PyTorch wheels don't match the accelerator arch.
    """
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        # Probe that kernels actually work on this GPU (not just "cuda available").
        x = torch.zeros(1, device="cuda")
        y = x + 1
        torch.cuda.synchronize()
        del x, y
        torch.cuda.empty_cache()
        return torch.device("cuda")
    except Exception as exc:
        print("CUDA probe failed → using CPU. Reason:", type(exc).__name__, str(exc)[:200])
        return torch.device("cpu")

DEVICE = pick_device(FORCE_CPU)
print("DEVICE:", DEVICE)
if DEVICE.type == "cpu" and torch.cuda.is_available():
    print("Note: CUDA is visible but unusable with this PyTorch build. DeepProbLog will run on CPU.")

if Path("/kaggle/working").is_dir():
    OUTPUT_DIR = Path("/kaggle/working/processed")
else:
    OUTPUT_DIR = Path("../notebooks/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if SMOKE_TEST:
    print("=" * 64)
    print("SMOKE_TEST=True — tiny wiring test on a small sample.")
    print("=" * 64)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


## 4) Find the four CBIS CSVs

**What:** search Kaggle input, then local `data/`, for the official mass and calc description files.

**Why:** the files are not always at a fixed path. Walking the tree (skipping `jpeg/`) keeps this cell working on Kaggle and on a laptop.


In [ ]:
def find_csv_paths(root, filenames):
    root = Path(root)
    found, remaining = {}, set(filenames.values())
    for dirpath, dirnames, files in os.walk(root):
        dirnames[:] = [d for d in dirnames if d.lower() != "jpeg"]
        for fname in files:
            if fname in remaining:
                key = next(k for k, v in filenames.items() if v == fname)
                found[key] = Path(dirpath) / fname
                remaining.discard(fname)
        if not remaining:
            break
    missing = [k for k in filenames if k not in found]
    if missing:
        raise FileNotFoundError(f"Missing {missing} under {root}")
    return found


def resolve_data_root():
    if KAGGLE_INPUT.is_dir():
        return KAGGLE_INPUT, "kaggle_input"
    # other common Kaggle mount
    for p in Path("/kaggle/input").glob("*cbis*") if Path("/kaggle/input").is_dir() else []:
        if p.is_dir():
            return p, "kaggle_input_scan"
    if LOCAL_DATA.is_dir() and all((LOCAL_DATA / fn).exists() for fn in CSV_FILENAMES.values()):
        return LOCAL_DATA, "local_data"
    try:
        import kagglehub
        downloaded = Path(kagglehub.dataset_download(DATASET_HANDLE))
        return downloaded, "kagglehub_download"
    except Exception as exc:
        raise FileNotFoundError("No CBIS-DDSM data found") from exc


DATA_ROOT, DATA_SOURCE = resolve_data_root()
CSV_PATHS = find_csv_paths(DATA_ROOT, CSV_FILENAMES)
print(f"Data source: {DATA_SOURCE}\nRoot: {DATA_ROOT}")
for k, v in CSV_PATHS.items():
    print(f"  {k}: {v}")


## 5) Merge mass+calc, fix leakage, split by patient

**What:** concatenate mass and calcification tables, assign overlapping patients entirely to train, then cut a patient-disjoint validation set.

**Why:** after merging, some patients appear in both official train and test. A patient-level split is required so the same woman is never in train and test.


In [ ]:
STRING_COLUMNS = [
    "patient_id", "left or right breast", "image view", "abnormality type",
    "mass shape", "mass margins", "calc type", "calc distribution", "pathology",
    "image file path", "cropped image file path", "ROI mask file path",
]


def normalize_columns(df):
    out = df.copy()
    out.columns = [c.strip() for c in out.columns]
    if "breast density" in out.columns:
        out = out.rename(columns={"breast density": "breast_density"})
    for col in STRING_COLUMNS:
        if col in out.columns:
            out[col] = out[col].astype(str).str.strip().replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return out


def add_meta(df, kind, official_split):
    out = df.copy()
    out["abnormality_kind"] = kind
    out["official_split"] = official_split
    return out


def sample_patients(df, n_patients, seed):
    patients = df["patient_id"].dropna().unique()
    if len(patients) <= n_patients:
        return df.copy()
    chosen = set(random.Random(seed).sample(list(patients), n_patients))
    return df[df["patient_id"].isin(chosen)].copy()


def resolve_cross_split_patient_leakage(train_df, test_df):
    """CBIS mass/calc official CSVs can place the same patient in train AND test
    (e.g. mass in train, calc in test). Keep ALL rows for those patients in train
    and drop them from test so splits stay patient-disjoint.
    """
    train_p = set(train_df["patient_id"].dropna())
    test_p = set(test_df["patient_id"].dropna())
    overlap = train_p & test_p
    if not overlap:
        print("No cross-split patient overlap.")
        return train_df, test_df

    print(f"Found {len(overlap)} patients in BOTH official train and test after merge.")
    print("  Assigning all their rows to TRAIN and removing them from TEST.")
    leaked_test_rows = test_df[test_df["patient_id"].isin(overlap)].copy()
    train_fixed = pd.concat([train_df, leaked_test_rows], ignore_index=True)
    test_fixed = test_df[~test_df["patient_id"].isin(overlap)].reset_index(drop=True)
    # drop duplicate rows if any exact duplicates appear
    train_fixed = train_fixed.drop_duplicates().reset_index(drop=True)
    return train_fixed, test_fixed


raw = {k: pd.read_csv(v) for k, v in CSV_PATHS.items()}
mass_train = add_meta(normalize_columns(raw["mass_train"]), "mass", "train")
mass_test = add_meta(normalize_columns(raw["mass_test"]), "mass", "test")
calc_train = add_meta(normalize_columns(raw["calc_train"]), "calcification", "train")
calc_test = add_meta(normalize_columns(raw["calc_test"]), "calcification", "test")

df_train_all = pd.concat([mass_train, calc_train], ignore_index=True)
df_test_all = pd.concat([mass_test, calc_test], ignore_index=True)
print("Official train:", len(df_train_all), "rows |", df_train_all["patient_id"].nunique(), "patients")
print("Official test: ", len(df_test_all), "rows |", df_test_all["patient_id"].nunique(), "patients")

df_train_all, df_test_all = resolve_cross_split_patient_leakage(df_train_all, df_test_all)
print("After leakage fix train:", len(df_train_all), "rows |", df_train_all["patient_id"].nunique(), "patients")
print("After leakage fix test: ", len(df_test_all), "rows |", df_test_all["patient_id"].nunique(), "patients")

if SMOKE_TEST:
    df_train_all = sample_patients(df_train_all, SMOKE_N_PATIENTS, RANDOM_SEED)
    df_test_all = sample_patients(df_test_all, SMOKE_N_PATIENTS, RANDOM_SEED + 1)
    print("After smoke sampling:", len(df_train_all), "train rows |", len(df_test_all), "test rows")

gss = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
tr_idx, va_idx = next(gss.split(df_train_all, groups=df_train_all["patient_id"]))
df_train = df_train_all.iloc[tr_idx].reset_index(drop=True)
df_val = df_train_all.iloc[va_idx].reset_index(drop=True)
df_test = df_test_all.reset_index(drop=True)

tr_p, va_p, te_p = set(df_train.patient_id), set(df_val.patient_id), set(df_test.patient_id)
print("overlap train∩val:", len(tr_p & va_p), "| train∩test:", len(tr_p & te_p), "| val∩test:", len(va_p & te_p))
assert tr_p.isdisjoint(va_p), "Train/val patient leakage"
assert tr_p.isdisjoint(te_p), "Train/test patient leakage"
assert va_p.isdisjoint(te_p), "Val/test patient leakage"
print("Leakage checks passed.")
for name, df in (("train", df_train), ("val", df_val), ("test", df_test)):
    print(f"{name:5s}: {len(df):>5} rows | {df['patient_id'].nunique():>4} patients")


## 6) Save the split tables

**What:** write `cbis_train.csv`, `cbis_val.csv`, `cbis_test.csv`, and a patient-to-split map.

**Why:** so you can inspect exactly which cases this run used.


In [ ]:
# Save processed artifacts (same session will also train on these frames)
patient_splits = []
for split_name, frame in (("train", df_train), ("val", df_val), ("test", df_test)):
    for pid in sorted(frame["patient_id"].dropna().unique()):
        patient_splits.append({"patient_id": pid, "split": split_name})
pd.DataFrame(patient_splits).to_csv(OUTPUT_DIR / "cbis_patient_splits.csv", index=False)
df_train.to_csv(OUTPUT_DIR / "cbis_train.csv", index=False)
df_val.to_csv(OUTPUT_DIR / "cbis_val.csv", index=False)
df_test.to_csv(OUTPUT_DIR / "cbis_test.csv", index=False)

meta = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "smoke_test": SMOKE_TEST,
    "smoke_n_patients": SMOKE_N_PATIENTS if SMOKE_TEST else None,
    "val_size": VAL_SIZE,
    "random_seed": RANDOM_SEED,
    "data_source": DATA_SOURCE,
    "data_root": str(DATA_ROOT),
    "counts": {
        "train_rows": len(df_train),
        "val_rows": len(df_val),
        "test_rows": len(df_test),
        "train_patients": df_train["patient_id"].nunique(),
        "val_patients": df_val["patient_id"].nunique(),
        "test_patients": df_test["patient_id"].nunique(),
    },
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))
print("Saved prep outputs to", OUTPUT_DIR.resolve())


## 7) Map CSV descriptors to concepts

**What:** convert `mass shape`, `mass margins`, `calc type`, and `abnormality_kind` into the symbols used in the Prolog program.

**Why:** DeepProbLog queries `shape(X, irregular)`, not the original CBIS strings. Any calcification is **not** treated as malignant; only suspicious types map to `suspicious`.


In [ ]:
def tokenize_descriptor(value):
    if pd.isna(value):
        return []
    text = str(value).strip().upper()
    if text in {"", "NAN", "NONE"}:
        return []
    for sep in ("-", "/", "_", " "):
        text = text.replace(sep, " ")
    return [t for t in text.split() if t]


def map_shape(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if any(t in {"IRREGULAR", "ARCHITECTURAL", "DISTORTION"} for t in tokens):
        return "irregular"
    if any(t in {"ROUND", "OVAL", "LOBULATED"} for t in tokens):
        return "regular"
    return "unknown"


def map_margin(value):
    tokens = tokenize_descriptor(value)
    if not tokens:
        return "unknown"
    if "SPICULATED" in tokens:
        return "spiculated"
    if "CIRCUMSCRIBED" in tokens:
        return "circumscribed"
    return "other"


def map_calc(row):
    # Any calcification present is NOT a malignancy label. Most CBIS calc cases are benign.
    if str(row.get("abnormality_kind", "")).lower() != "calcification":
        return "absent"
    tokens = tokenize_descriptor(row.get("calc type"))
    if any(t in {"PLEOMORPHIC", "AMORPHOUS"} for t in tokens) or (
        "FINE" in tokens and "LINEAR" in tokens
    ) or "BRANCHING" in tokens:
        return "suspicious"
    return "typically_benign"


def add_concepts(df):
    out = df.copy()
    out["shape_concept"] = out["mass shape"].map(map_shape) if "mass shape" in out.columns else "unknown"
    out["margin_concept"] = out["mass margins"].map(map_margin) if "mass margins" in out.columns else "unknown"
    out["calc_concept"] = out.apply(map_calc, axis=1)
    out["kind_concept"] = out["abnormality_kind"].map(
        lambda x: "calcification" if str(x).strip().lower() == "calcification" else "mass"
    )
    out["malignant"] = out["pathology"].map(lambda x: 1.0 if str(x).strip().upper() == "MALIGNANT" else 0.0)
    return out


df_train = add_concepts(df_train)
df_val = add_concepts(df_val)
df_test = add_concepts(df_test)
display(df_train[["patient_id", "abnormality_kind", "shape_concept", "margin_concept", "calc_concept", "malignant"]].sample(10))
print("calc_concept counts:\n", df_train["calc_concept"].value_counts())
print("malignant rate train/val/test:",
      float(df_train["malignant"].mean()),
      float(df_val["malignant"].mean()),
      float(df_test["malignant"].mean()))


## 8) Load validation and test images only

Train images are skipped. Validation and test keep the result_4 order.


In [ ]:
import os
import time
import concurrent.futures
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

NUM_WORKERS = min(8, (os.cpu_count() or 4) * 2)  # I/O-bound → oversubscribe cores a bit
THUMBNAIL_FOR_SCORE = 256  # small copy used only to pick photo vs mask, not for training


def find_jpeg_root():
    candidates = [DATA_ROOT / "jpeg", KAGGLE_INPUT / "jpeg", LOCAL_DATA / "jpeg"]
    if Path("/kaggle/input").is_dir():
        candidates.extend(p for p in Path("/kaggle/input").rglob("jpeg") if p.is_dir())
    for p in candidates:
        if p.is_dir():
            return p
    return None


JPEG_ROOT = find_jpeg_root()
print("JPEG root:", JPEG_ROOT)
print("Tensors will be placed on:", DEVICE)
print(f"Parallel image loading workers: {NUM_WORKERS}")

# One fixed patch per image. Multi-patch bags are NOT used here: each extra
# patch would multiply DeepProbLog ExactEngine queries.
def build_patch_transform(train=False):
    """ROI crop -> IMAGE_SIZE x IMAGE_SIZE tensor.

    CBIS 'cropped' JPEGs are already lesion-centered patches of the full mammo.
    PATCH_MODE controls how we cut a fixed square from that ROI without always
    stretching the whole box (which destroys fine margin detail).
    """
    if PATCH_MODE == "resize":
        return transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
        ])
    # center / random: scale so short side is at least IMAGE_SIZE, then crop
    if train and PATCH_MODE == "random":
        return transforms.Compose([
            transforms.RandomResizedCrop(
                IMAGE_SIZE, scale=(0.7, 1.0), ratio=(0.9, 1.1),
                interpolation=transforms.InterpolationMode.BILINEAR,
            ),
            transforms.ToTensor(),
        ])
    return transforms.Compose([
        transforms.Resize(IMAGE_SIZE, interpolation=transforms.InterpolationMode.BILINEAR),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
    ])

tfm_train = build_patch_transform(train=True)
tfm_eval = build_patch_transform(train=False)
print(f"PATCH_MODE={PATCH_MODE!r}  IMAGE_SIZE={IMAGE_SIZE}  "
      f"(train={'RandomResizedCrop' if PATCH_MODE=='random' else PATCH_MODE}, "
      f"val/test={'center crop' if PATCH_MODE!='resize' else 'resize'})")


def clean_relpath(value):
    if pd.isna(value):
        return None
    text = str(value).strip().strip('"').split("\n")[0].strip()
    return text or None


_JPEG_INDEX = None


def jpeg_folder_index():
    """Map folder names that actually contain images to their paths.

    The awsaf49 jpeg tree is not the CSV DICOM path. Images usually live in a
    folder named by the series UID (3rd path component), as 1-1.jpg / 1-2.jpg.
    """
    global _JPEG_INDEX
    if _JPEG_INDEX is not None:
        return _JPEG_INDEX
    index = {}
    if JPEG_ROOT is None or not JPEG_ROOT.is_dir():
        _JPEG_INDEX = index
        return index
    t0 = time.perf_counter()
    for dirpath, _dirnames, filenames in os.walk(JPEG_ROOT):
        imgs = [f for f in filenames if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        if not imgs:
            continue
        folder = Path(dirpath)
        index[folder.name] = folder
    _JPEG_INDEX = index
    dt = time.perf_counter() - t0
    print(f"Indexed {len(index)} jpeg folders in {dt:.1f}s")
    print("example jpeg folders:", list(index)[:5])
    return index


def image_photo_score(path):
    """Higher score = more like a mammogram crop, less like a binary ROI mask.

    Scored on a small thumbnail: we only need the ranking between candidate
    files in the same folder, not exact statistics, so this is much cheaper
    than scoring the full-resolution image.
    """
    with Image.open(path) as im:
        im = im.convert("L")
        im.thumbnail((THUMBNAIL_FOR_SCORE, THUMBNAIL_FOR_SCORE))
        arr = np.asarray(im, dtype=np.uint8)
    n_unique = int(len(np.unique(arr)))
    mid_gray = float(((arr > 15) & (arr < 240)).mean())
    return (n_unique, mid_gray, float(arr.std()))


def pick_image(folder, prefer_crop=True):
    folder = Path(folder)
    jpgs = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.jpeg")) + sorted(folder.glob("*.png"))
    if not jpgs:
        jpgs = sorted(folder.rglob("*.jpg")) + sorted(folder.rglob("*.jpeg")) + sorted(folder.rglob("*.png"))
    if not jpgs:
        return None
    if len(jpgs) == 1:
        return jpgs[0]
    # CBIS crop/mask folders contain 1-1.jpg and 1-2.jpg, and the CSV columns
    # are often swapped. Prefer the file with many gray levels (photo), not a mask.
    return max(jpgs, key=image_photo_score)


def resolve_jpeg(row):
    if JPEG_ROOT is None:
        return None
    index = jpeg_folder_index()
    for col in ("cropped image file path", "image file path"):
        rel = clean_relpath(row.get(col))
        if not rel:
            continue
        parts = [p for p in rel.replace("\\", "/").split("/") if p]
        if not parts:
            continue
        prefer_crop = col.startswith("cropped")
        exact = JPEG_ROOT / rel.replace(".dcm", ".jpg")
        if exact.is_file():
            return exact
        keys = []
        # Do not use Path.stem here. DICOM UIDs contain dots, so stem()
        # would chop the series id and never match the jpeg folder name.
        for part in parts:
            key = part[:-4] if part.lower().endswith(".dcm") else part
            if key:
                keys.append(key)
        seen = set()
        for key in keys:
            if not key or key in seen:
                continue
            seen.add(key)
            folder = index.get(key)
            if folder is None and (JPEG_ROOT / key).is_dir():
                folder = JPEG_ROOT / key
            if folder is None:
                continue
            img = pick_image(folder, prefer_crop=prefer_crop)
            if img is not None:
                return img
    return None


def _to_device(t):
    """Move tensor to DEVICE; if CUDA fails mid-run, switch globally to CPU."""
    global DEVICE
    try:
        return t.to(DEVICE)
    except Exception as exc:
        print("Device move failed (", type(exc).__name__, ") → forcing CPU for the rest of the run.")
        DEVICE = torch.device("cpu")
        return t.to(DEVICE)


def concept_placeholder(row):
    """Used only when a JPEG is missing. These placeholders are for a smoke test, not a real-image run."""
    shape_c = row.get("shape_concept", "unknown")
    margin_c = row.get("margin_concept", "unknown")
    calc_c = row.get("calc_concept", "absent")
    t = torch.zeros(1, IMAGE_SIZE, IMAGE_SIZE)
    third = IMAGE_SIZE // 3
    t[0, :third, :] = {"irregular": 1.0, "regular": 0.55, "unknown": 0.15}.get(shape_c, 0.15)
    t[0, third:2 * third, :] = {"spiculated": 1.0, "circumscribed": 0.55, "other": 0.35, "unknown": 0.15}.get(margin_c, 0.15)
    t[0, 2 * third:, :] = {"suspicious": 1.0, "typically_benign": 0.55, "absent": 0.15}.get(calc_c, 0.15)
    return t


def load_tensor(row, train=False):
    path = resolve_jpeg(row)
    if path is not None and path.is_file():
        with Image.open(path) as img:
            tfm = tfm_train if train else tfm_eval
            tensor = _to_device(tfm(img.convert("L")))
            return tensor, str(path), False
    if not USE_SYNTHETIC_IF_MISSING:
        rel = clean_relpath(row.get("cropped image file path")) or clean_relpath(row.get("image file path"))
        raise FileNotFoundError(
            f"No JPEG for patient {row.get('patient_id')}. csv_path={rel!r}. "
            f"JPEG_ROOT={JPEG_ROOT}. Check the 'Indexed N jpeg folders' line above."
        )
    return _to_device(concept_placeholder(row)), None, True


def materialize_split(df, n_limit=None, split_name="split", train=False):
    if n_limit is not None:
        df = df.head(n_limit).copy()
    rows = list(df.iterrows())
    n = len(rows)
    tensors = [None] * n
    paths = [None] * n
    synths = [False] * n
    t_start = time.perf_counter()

    def _worker(item):
        i, (_, row) = item
        tensor, path, synthetic = load_tensor(row, train=train)
        return i, tensor, path, synthetic

    n_synth = 0
    with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_WORKERS) as ex:
        futures = ex.map(_worker, enumerate(rows))
        for i, tensor, path, synthetic in tqdm(futures, total=n, desc=f"Materializing {split_name}", unit="img"):
            tensors[i] = tensor
            paths[i] = path
            synths[i] = synthetic
            n_synth += int(synthetic)

    elapsed = time.perf_counter() - t_start
    avg_ms = (elapsed / max(n, 1)) * 1000
    print(f"  loaded {n} tensors ({n_synth} synthetic) on {DEVICE} "
          f"in {elapsed:.1f}s (avg {avg_ms:.0f} ms/image, {NUM_WORKERS} workers)")

    records = []
    for (_, row), path, synthetic in zip(rows, paths, synths):
        rec = row.to_dict()
        rec["resolved_path"] = path
        rec["synthetic"] = synthetic
        records.append(rec)
    return tensors, pd.DataFrame(records)


# Build the folder index once, up front, before the thread pool starts —
# avoids every worker thread racing to build it redundantly on first access.
jpeg_folder_index()

limit = SMOKE_N_SAMPLES if SMOKE_TEST else None
t_all_start = time.perf_counter()
print("Eval notebook: train images are not loaded")
train_tensors = []
print("Materializing val");   val_tensors, df_val = materialize_split(df_val, limit, split_name="val", train=False)
print("Materializing test");  test_tensors, df_test = materialize_split(df_test, limit, split_name="test", train=False)
print(f"Total materialization time: {time.perf_counter() - t_all_start:.1f}s")
if int(df_test["synthetic"].sum()) or int(df_val["synthetic"].sum()):
    raise RuntimeError("Synthetic images were loaded. Check JPEG_ROOT before trusting any number.")
print(f"Real images: val {len(df_val)}, test {len(df_test)}. Augmentation skipped.")


## 9) Network definitions

Same classes as result_4. Nothing is trained in this cell.


In [ ]:
import time
# ImageNet normalization (applied after repeating grayscale -> 3 channels)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def _prepare_input(x, device, three_channel=False):
    """DeepProbLog may pass CPU / unbatched tensors — normalize shape and device."""
    if not torch.is_tensor(x):
        x = torch.as_tensor(x)
    x = x.to(device, dtype=torch.float32)
    if x.dim() == 3:  # [C,H,W] -> [1,C,H,W]
        x = x.unsqueeze(0)
    if three_channel:
        if x.size(1) == 1:
            x = x.repeat(1, 3, 1, 1)
        mean = x.new_tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
        std = x.new_tensor(IMAGENET_STD).view(1, 3, 1, 1)
        x = (x - mean) / std
    return x


class ConceptCNN(nn.Module):
    """Tiny from-scratch CNN (BACKBONE='tiny'). Kept for smoke tests / ablation."""

    def __init__(self, n_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(True),
            nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Linear(64 * 4 * 4, n_classes)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x, return_prob=True):
        x = _prepare_input(x, next(self.parameters()).device, three_channel=False)
        logits = self.classifier(self.net(x).flatten(1))
        return self.softmax(logits) if return_prob else logits


class ResNetBackbone(nn.Module):
    """ImageNet ResNet18 trunk -> 512-d pooled features."""

    def __init__(self, pretrained=True):
        super().__init__()
        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        net = models.resnet18(weights=weights)
        net.fc = nn.Identity()
        self.net = net
        self.out_dim = 512

    def forward(self, x):
        return self.net(x)


class ConceptHead(nn.Module):
    """One DeepProbLog neural predicate: shared ResNet trunk + linear head + softmax.

    Only the head that gets register_backbone=True owns the trunk as a submodule
    (and therefore in parameters() / its Adam). Other heads hold a plain reference
    so DeepProbLog's optimizer.step() — which steps every Network each batch —
    does not Adam-update the shared weights four times.
    """

    def __init__(self, backbone: ResNetBackbone, n_classes: int, register_backbone: bool = False):
        super().__init__()
        self.register_backbone = register_backbone
        if register_backbone:
            self.backbone = backbone
        else:
            object.__setattr__(self, "_backbone_ref", backbone)
        self.head = nn.Linear(backbone.out_dim, n_classes)
        self.softmax = nn.Softmax(dim=1)

    def _backbone(self):
        return self.backbone if self.register_backbone else self._backbone_ref

    def forward(self, x, return_prob=True):
        bb = self._backbone()
        x = _prepare_input(x, next(bb.parameters()).device, three_channel=True)
        feats = bb(x)
        logits = self.head(feats)
        return self.softmax(logits) if return_prob else logits


def build_concept_nets(device=DEVICE):
    """Return (kind, shape, margin, calc, backbone_or_None) on device."""
    if BACKBONE == "tiny":
        kind = ConceptCNN(2).to(device)
        shape = ConceptCNN(3).to(device)
        margin = ConceptCNN(4).to(device)
        calc = ConceptCNN(3).to(device)
        return kind, shape, margin, calc, None

    if BACKBONE != "resnet18":
        raise ValueError(f"Unknown BACKBONE={BACKBONE!r}; use 'resnet18' or 'tiny'")

    backbone = ResNetBackbone(pretrained=False).to(device)
    kind = ConceptHead(backbone, 2, register_backbone=True).to(device)
    shape = ConceptHead(backbone, 3, register_backbone=False).to(device)
    margin = ConceptHead(backbone, 4, register_backbone=False).to(device)
    calc = ConceptHead(backbone, 3, register_backbone=False).to(device)
    return kind, shape, margin, calc, backbone


def set_backbone_trainable(backbone, trainable: bool):
    if backbone is None:
        return
    for p in backbone.parameters():
        p.requires_grad = bool(trainable)
    print(f"Backbone requires_grad={trainable}")


def make_network_optimizers(kind_net, shape_net, margin_net, calc_net, backbone, lr):
    """Adam per DeepProbLog Network. Shared trunk only in kind_net's optimizer."""
    if backbone is None:
        return (
            torch.optim.Adam(kind_net.parameters(), lr=lr),
            torch.optim.Adam(shape_net.parameters(), lr=lr),
            torch.optim.Adam(margin_net.parameters(), lr=lr),
            torch.optim.Adam(calc_net.parameters(), lr=lr),
        )
    backbone_lr = lr * BACKBONE_LR_MULT
    bb_params = [p for p in backbone.parameters() if p.requires_grad]
    kind_groups = [{"params": kind_net.head.parameters(), "lr": lr}]
    if bb_params:
        kind_groups.insert(0, {"params": bb_params, "lr": backbone_lr})
    kind_opt = torch.optim.Adam(kind_groups)
    shape_opt = torch.optim.Adam(shape_net.head.parameters(), lr=lr)
    margin_opt = torch.optim.Adam(margin_net.head.parameters(), lr=lr)
    calc_opt = torch.optim.Adam(calc_net.head.parameters(), lr=lr)
    return kind_opt, shape_opt, margin_opt, calc_opt



# Ordered label lists must match PROGRAM_FALLBACK / DeepProbLog class order.
KIND_LABELS = ["mass", "calcification"]
SHAPE_LABELS = ["regular", "irregular", "unknown"]
MARGIN_LABELS = ["circumscribed", "spiculated", "other", "unknown"]
CALC_LABELS = ["absent", "typically_benign", "suspicious"]


def label_to_index(series, labels):
    idx = {lab: i for i, lab in enumerate(labels)}
    out = []
    for v in series:
        key = str(v).strip()
        if key not in idx:
            key = "unknown" if "unknown" in idx else labels[0]
        out.append(idx[key])
    return torch.tensor(out, dtype=torch.long)


def inverse_freq_weights(y_indices, n_classes):
    """Inverse-frequency class weights (normalized to mean 1)."""
    counts = torch.bincount(y_indices.cpu(), minlength=n_classes).float().clamp(min=1.0)
    w = counts.sum() / (n_classes * counts)
    return (w / w.mean()).to(DEVICE)


def concept_agree_from_logits(logits, y):
    return float((logits.argmax(dim=1) == y).float().mean().item())


def _head_param_list(kind_net, shape_net, margin_net, calc_net, backbone):
    if backbone is None:
        return (
            list(kind_net.parameters()) + list(shape_net.parameters())
            + list(margin_net.parameters()) + list(calc_net.parameters())
        )
    return (
        list(kind_net.head.parameters()) + list(shape_net.head.parameters())
        + list(margin_net.head.parameters()) + list(calc_net.head.parameters())
    )


def run_concept_warmup(kind_net, shape_net, margin_net, calc_net, backbone,
                       train_tensors, df_train, val_tensors=None, df_val=None):
    """Supervised CE warm-up on CSV concepts before DeepProbLog.

    Inverse-frequency class weights (esp. margin/shape). Freezes ResNet trunk for
    CONCEPT_WARMUP_FREEZE_EPOCHS, then fine-tunes at BACKBONE_LR_MULT * LR.
    """
    if CONCEPT_WARMUP_EPOCHS <= 0:
        print("CONCEPT_WARMUP_EPOCHS=0 — skipping supervised concept warm-up.")
        return None

    print(f"\n=== Concept CE warm-up: {CONCEPT_WARMUP_EPOCHS} epochs, "
          f"batch={CONCEPT_WARMUP_BATCH}, freeze_first={CONCEPT_WARMUP_FREEZE_EPOCHS} ===")

    df_w = globals().get("df_train_orig", df_train)

    y_kind = label_to_index(df_train["kind_concept"], KIND_LABELS)
    y_shape = label_to_index(df_train["shape_concept"], SHAPE_LABELS)
    y_margin = label_to_index(df_train["margin_concept"], MARGIN_LABELS)
    y_calc = label_to_index(df_train["calc_concept"], CALC_LABELS)

    w_kind = inverse_freq_weights(label_to_index(df_w["kind_concept"], KIND_LABELS), len(KIND_LABELS))
    w_shape = inverse_freq_weights(label_to_index(df_w["shape_concept"], SHAPE_LABELS), len(SHAPE_LABELS))
    w_margin = inverse_freq_weights(label_to_index(df_w["margin_concept"], MARGIN_LABELS), len(MARGIN_LABELS))
    w_calc = inverse_freq_weights(label_to_index(df_w["calc_concept"], CALC_LABELS), len(CALC_LABELS))
    print("Class weights kind :", [round(float(x), 3) for x in w_kind])
    print("Class weights shape:", [round(float(x), 3) for x in w_shape])
    print("Class weights margin:", [round(float(x), 3) for x in w_margin])
    print("Class weights calc :", [round(float(x), 3) for x in w_calc])

    crit_kind = nn.CrossEntropyLoss(weight=w_kind)
    crit_shape = nn.CrossEntropyLoss(weight=w_shape)
    crit_margin = nn.CrossEntropyLoss(weight=w_margin)
    crit_calc = nn.CrossEntropyLoss(weight=w_calc)

    x_cpu = torch.stack([t.detach().cpu() if torch.is_tensor(t) else t for t in train_tensors])
    from torch.utils.data import TensorDataset, DataLoader as TorchLoader
    ds = TensorDataset(x_cpu, y_kind, y_shape, y_margin, y_calc)
    loader = TorchLoader(
        ds, batch_size=CONCEPT_WARMUP_BATCH, shuffle=True,
        generator=torch.Generator().manual_seed(RANDOM_SEED),
    )

    def build_opt(freeze_backbone):
        set_backbone_trainable(backbone, (not freeze_backbone) if backbone is not None else True)
        head_params = _head_param_list(kind_net, shape_net, margin_net, calc_net, backbone)
        if backbone is None or freeze_backbone:
            return torch.optim.Adam(head_params, lr=LR)
        return torch.optim.Adam([
            {"params": [p for p in backbone.parameters() if p.requires_grad], "lr": LR * BACKBONE_LR_MULT},
            {"params": head_params, "lr": LR},
        ])

    freeze = backbone is not None and CONCEPT_WARMUP_FREEZE_EPOCHS > 0
    opt = build_opt(freeze_backbone=freeze)

    history = []
    t0 = time.perf_counter()
    for ep in range(CONCEPT_WARMUP_EPOCHS):
        if backbone is not None and ep == CONCEPT_WARMUP_FREEZE_EPOCHS and CONCEPT_WARMUP_FREEZE_EPOCHS > 0:
            print(f"  warm-up: unfreezing backbone at epoch {ep + 1}")
            freeze = False
            opt = build_opt(freeze_backbone=False)

        kind_net.train(); shape_net.train(); margin_net.train(); calc_net.train()
        if backbone is not None:
            backbone.train()
        total, n = 0.0, 0
        for xb, yk, ys, ym, yc in loader:
            xb = xb.to(DEVICE)
            yk, ys, ym, yc = yk.to(DEVICE), ys.to(DEVICE), ym.to(DEVICE), yc.to(DEVICE)
            opt.zero_grad()
            loss = (
                crit_kind(kind_net(xb, return_prob=False), yk)
                + crit_shape(shape_net(xb, return_prob=False), ys)
                + crit_margin(margin_net(xb, return_prob=False), ym)
                + crit_calc(calc_net(xb, return_prob=False), yc)
            )
            loss.backward()
            opt.step()
            total += float(loss.item()) * len(yk)
            n += len(yk)
        row = {"epoch": ep + 1, "loss": total / max(n, 1), "frozen": freeze}
        print(f"  warm-up epoch {ep + 1}/{CONCEPT_WARMUP_EPOCHS}  loss={row['loss']:.4f}  frozen={freeze}")
        history.append(row)

    val_agree = {}
    if val_tensors is not None and df_val is not None and len(df_val) > 0:
        kind_net.eval(); shape_net.eval(); margin_net.eval(); calc_net.eval()
        if backbone is not None:
            backbone.eval()
        with torch.no_grad():
            vk = label_to_index(df_val["kind_concept"], KIND_LABELS).to(DEVICE)
            vs = label_to_index(df_val["shape_concept"], SHAPE_LABELS).to(DEVICE)
            vm = label_to_index(df_val["margin_concept"], MARGIN_LABELS).to(DEVICE)
            vc = label_to_index(df_val["calc_concept"], CALC_LABELS).to(DEVICE)
            ag_k = ag_s = ag_m = ag_c = 0.0
            bs = CONCEPT_WARMUP_BATCH
            for i0 in range(0, len(df_val), bs):
                batch = [val_tensors[i].detach().cpu() for i in range(i0, min(i0 + bs, len(df_val)))]
                xb = torch.stack(batch).to(DEVICE)
                sl = slice(i0, i0 + len(batch))
                ag_k += concept_agree_from_logits(kind_net(xb, return_prob=False), vk[sl]) * len(batch)
                ag_s += concept_agree_from_logits(shape_net(xb, return_prob=False), vs[sl]) * len(batch)
                ag_m += concept_agree_from_logits(margin_net(xb, return_prob=False), vm[sl]) * len(batch)
                ag_c += concept_agree_from_logits(calc_net(xb, return_prob=False), vc[sl]) * len(batch)
            nval = float(len(df_val))
            val_agree = {
                "kind": ag_k / nval, "shape": ag_s / nval,
                "margin": ag_m / nval, "calc": ag_c / nval,
            }
            print("  warm-up val agree@argmax: "
                  + ", ".join(f"{k}={v:.3f}" for k, v in val_agree.items()))

    ckpt_path = OUTPUT_DIR / "checkpoints" / f"concept_warmup_{BACKBONE}.pt"
    ckpt_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "backbone_name": BACKBONE,
        "kind_net": kind_net.state_dict(),
        "shape_net": shape_net.state_dict(),
        "margin_net": margin_net.state_dict(),
        "calc_net": calc_net.state_dict(),
        "history": history,
        "val_agree": val_agree,
    }
    if backbone is not None:
        payload["backbone"] = backbone.state_dict()
    torch.save(payload, ckpt_path)
    print(f"  warm-up done in {(time.perf_counter() - t0) / 60:.1f} min -> {ckpt_path}")
    return {"history": history, "val_agree": val_agree, "path": str(ckpt_path)}


def draw_concept_architecture(image_size=IMAGE_SIZE):
    """Draw shared-backbone (or tiny) concept nets feeding DeepProbLog."""
    print(f"BACKBONE={BACKBONE!r}  IMAGE_SIZE={image_size}  "
          f"FREEZE_BACKBONE_EPOCHS={FREEZE_BACKBONE_EPOCHS}")
    if BACKBONE == "resnet18":
        stages = [
            ("input 1ch->3ch", image_size, image_size, 3),
            ("ResNet stem /2", image_size // 2, image_size // 2, 64),
            ("layer1", image_size // 2, image_size // 2, 64),
            ("layer2 /2", image_size // 4, image_size // 4, 128),
            ("layer3 /2", image_size // 8, image_size // 8, 256),
            ("layer4 /2", image_size // 16, image_size // 16, 512),
            ("GAP -> 512-d", 1, 1, 512),
        ]
        print("ResNet18 spatial path:")
    else:
        h = w = int(image_size)
        stages = [("input (1 ch)", h, w, 1)]
        for name, ch in (("conv1 stride2", 16), ("conv2 stride2", 32), ("conv3 stride2", 64)):
            h = (h + 1) // 2
            w = (w + 1) // 2
            stages.append((f"{name} ({ch} ch)", h, w, ch))
        stages.append(("AdaptiveAvgPool2d(4)", 4, 4, 64))
        print("Tiny ConceptCNN spatial path:")
    for name, h, w, ch in stages:
        print(f"  {name:28s}  {h} x {w}  ch={ch}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
    ax = axes[0]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis("off")
    ax.set_title("DeepProbLog concept networks", fontsize=11)
    trunk_label = (
        f"shared ResNet18\n{image_size}x{image_size}"
        if BACKBONE == "resnet18"
        else f"tiny CNN\n{image_size}x{image_size}"
    )
    ax.add_patch(plt.Rectangle((0.3, 3.5), 2.4, 3.0, fill=True, facecolor="#e8eef5", edgecolor="black", lw=1.2))
    ax.text(1.5, 5.0, trunk_label, ha="center", va="center", fontsize=8)
    heads = [
        (4.2, 8.2, "kind_net\n2 classes"),
        (4.2, 6.0, "shape_net\n3 classes"),
        (4.2, 3.8, "margin_net\n4 classes"),
        (4.2, 1.6, "calc_net\n3 classes"),
    ]
    for x, y, label in heads:
        ax.add_patch(plt.Rectangle((x, y - 0.7), 2.6, 1.4, fill=True, facecolor="#dfece6", edgecolor="black", lw=1.2))
        ax.text(x + 1.3, y, label, ha="center", va="center", fontsize=8)
        ax.annotate("", xy=(x, y), xytext=(2.7, 5.0), arrowprops=dict(arrowstyle="->", color="black", lw=1.0))
    ax.add_patch(plt.Rectangle((7.4, 3.5), 2.2, 3.0, fill=True, facecolor="#f5e8e8", edgecolor="black", lw=1.2))
    ax.text(8.5, 5.0, "DeepProbLog\nrules\n-> P(malignant)", ha="center", va="center", fontsize=8)
    for _, y, _ in heads:
        ax.annotate("", xy=(7.4, 5.0), xytext=(6.8, y), arrowprops=dict(arrowstyle="->", color="black", lw=0.9))

    ax = axes[1]
    ax.set_xlim(0, 12)
    ax.set_ylim(0, 8)
    ax.axis("off")
    ax.set_title(f"Spatial path ({BACKBONE})", fontsize=11)
    draw_stages = stages[:6] if len(stages) > 6 else stages
    max_side = max(max(s[1], 1) for s in draw_stages)
    xs = np.linspace(0.3, 10.0, len(draw_stages))
    for (name, h, w, ch), x in zip(draw_stages, xs):
        side = 0.5 + 2.0 * (h / max_side)
        y0 = 4.0 - side / 2
        ax.add_patch(plt.Rectangle((x, y0), 1.5, side, fill=True, facecolor="#e8eef5", edgecolor="black", lw=1.0))
        ax.text(x + 0.75, y0 + side / 2, f"{h}x{w}\n{ch}", ha="center", va="center", fontsize=6.5)
        ax.text(x + 0.75, y0 - 0.3, name.split()[0], ha="center", va="top", fontsize=6)
    ax.text(5.5, 7.3, "ImageNet init + shared trunk  |  each head: Linear -> Softmax", ha="center", fontsize=7.5)
    fig.tight_layout()
    plt.show()


draw_concept_architecture(IMAGE_SIZE)
print("Networks factory ready. Fresh copies are created for the training run.")

from problog.logic import Term, Constant
from deepproblog.dataset import Dataset, DataLoader, QueryDataset
from deepproblog.query import Query


class TensorSource:
    def __init__(self, tensors, device=None):
        self.tensors = tensors
        self.device = device

    def __getitem__(self, item):
        t = self.tensors[int(item[0])]
        if self.device is not None:
            t = t.to(self.device)
        return t


class BreastQueryDataset(Dataset):
    def __init__(self, split_name, df):
        self.split_name = split_name
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def _tensor_term(self, i):
        return Term("tensor", Term(self.split_name, Constant(i)))

    def to_query(self, i):
        row = self.df.iloc[i]
        return Query(Term("malignant", self._tensor_term(i)), p=float(row["malignant"]))

    def concept_queries(self):
        qs = []
        for i, row in self.df.iterrows():
            img = self._tensor_term(i)
            qs.append(Query(Term("shape", img, Term(row["shape_concept"])), p=1.0))
            qs.append(Query(Term("margin", img, Term(row["margin_concept"])), p=1.0))
            qs.append(Query(Term("calc", img, Term(row["calc_concept"])), p=1.0))
            qs.append(Query(Term("lesion_kind", img, Term(row["kind_concept"])), p=1.0))
        return qs


print("Dataset classes ready. Query datasets are built inside the training function so grid search can rebuild them.")


## 10) Score the paper checkpoints

Stage 2 is checked first. Stage 1 is scored only after those checks pass. No weights are updated.


In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import confusion_matrix, roc_auc_score
from tqdm.auto import tqdm
from deepproblog.engines import ExactEngine
from deepproblog.model import Model
from deepproblog.network import Network

PAPER_AGREE = {"kind": 0.905, "shape": 0.723, "margin": 0.701, "calc": 0.820}
HEADS = (
    ("kind", "kind_net", KIND_LABELS, "kind_concept"),
    ("shape", "shape_net", SHAPE_LABELS, "shape_concept"),
    ("margin", "margin_net", MARGIN_LABELS, "margin_concept"),
    ("calc", "calc_net", CALC_LABELS, "calc_concept"),
)


def find_weight(name):
    roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path("."), OUTPUT_DIR]
    hits = []
    for root in roots:
        if not root.exists():
            continue
        if root.is_file() and root.name == name:
            hits.append(root)
            continue
        for dirpath, dirnames, filenames in os.walk(root):
            dirnames[:] = [d for d in dirnames if d.lower() != "jpeg"]
            if name in filenames:
                hits.append(Path(dirpath) / name)
    if not hits:
        raise FileNotFoundError(f"Could not find {name}. Upload notebooks/res4/weights as a Kaggle dataset.")
    hits.sort(key=lambda p: (0 if "weights" in p.parts else 1, len(str(p))))
    print(f"{name}: {hits[0]}")
    return hits[0]


WARMUP_PATH = find_weight("concept_warmup_resnet18.pt")
STAGE2_PATH = find_weight("deepproblog_model.pth")
TEST_PROB_PATH = find_weight("test_prob.npy")


def extract_prob(result):
    if result is None:
        return 0.0
    mapping = result.result if hasattr(result, "result") else {}
    if not mapping:
        return 0.0
    vals = []
    for value in mapping.values():
        if hasattr(value, "detach"):
            vals.append(float(value.detach().cpu().item() if value.numel() == 1 else value.detach().cpu().mean()))
        else:
            vals.append(float(value))
    return float(sum(vals) / len(vals)) if vals else 0.0


def metrics_at(y_true, y_prob, thr):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    y_pred = (y_prob >= thr).astype(int)
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    tn = int(((y_pred == 0) & (y_true == 0)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())

    def f1(a, b, c):
        prec = a / max(a + b, 1)
        rec = a / max(a + c, 1)
        return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

    return {
        "threshold": float(thr),
        "accuracy": (tp + tn) / len(y_true),
        "sensitivity": tp / max(tp + fn, 1),
        "specificity": tn / max(tn + fp, 1),
        "f1_macro": 0.5 * (f1(tp, fp, fn) + f1(tn, fn, fp)),
        "roc_auc": float(roc_auc_score(y_true, y_prob)),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def choose_threshold(y_true, y_prob, target_sens=0.80):
    grid = np.round(np.arange(0.05, 0.96, 0.01), 2)
    rows = [metrics_at(y_true, y_prob, t) for t in grid]
    ok = [r for r in rows if r["sensitivity"] >= target_sens]
    if not ok:
        youden = max(rows, key=lambda r: r["sensitivity"] + r["specificity"] - 1)
        print(f"No threshold reaches sensitivity >= {target_sens}; falling back to Youden {youden['threshold']:.2f}")
        return youden["threshold"], False
    best_spec = max(r["specificity"] for r in ok)
    chosen = next(r for r in ok if r["specificity"] == best_spec)
    return chosen["threshold"], True


def load_warmup(nets, path):
    kind_net, shape_net, margin_net, calc_net, backbone = nets
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    kind_net.load_state_dict(ckpt["kind_net"])
    shape_net.load_state_dict(ckpt["shape_net"])
    margin_net.load_state_dict(ckpt["margin_net"])
    calc_net.load_state_dict(ckpt["calc_net"])
    backbone.load_state_dict(ckpt["backbone"])
    print("Loaded Stage 1. Warm-up val agree@argmax:",
          {k: round(v, 3) for k, v in ckpt.get("val_agree", {}).items()})


def build_eval_model(nets):
    kind_net, shape_net, margin_net, calc_net, _backbone = nets
    for net in nets:
        net.eval()
    program_path = OUTPUT_DIR / "breast_diagnosis.pl"
    program_path.write_text(PROGRAM_FALLBACK + "\n")
    trained = Model(str(program_path), [
        Network(shape_net, "shape_net", batching=True),
        Network(margin_net, "margin_net", batching=True),
        Network(calc_net, "calc_net", batching=True),
        Network(kind_net, "kind_net", batching=True),
    ])
    trained.set_engine(ExactEngine(trained), cache=False)
    trained.add_tensor_source("val", TensorSource(val_tensors, device=DEVICE))
    trained.add_tensor_source("test", TensorSource(test_tensors, device=DEVICE))
    return trained


def query_prob(trained, split_name, i):
    q = Query(Term("malignant", Term("tensor", Term(split_name, Constant(i)))))
    with torch.no_grad():
        solved = trained.solve([q])
    return extract_prob(solved[0]) if solved else 0.0


def score_split(trained, split_name, df):
    y_true, y_prob = [], []
    for i in tqdm(range(len(df)), desc=f"Stage query {split_name}", unit="img"):
        y_true.append(int(df.iloc[i]["malignant"]))
        y_prob.append(query_prob(trained, split_name, i))
    return np.asarray(y_true), np.asarray(y_prob, dtype=float)


@torch.no_grad()
def softmax_batches(net, tensors):
    net.eval()
    out = []
    for i0 in range(0, len(tensors), 32):
        batch = torch.stack([tensors[i].detach().cpu() for i in range(i0, min(i0 + 32, len(tensors)))])
        out.append(net(batch.to(DEVICE), return_prob=True).detach().cpu())
    return torch.cat(out, 0).numpy()


def concept_report(nets_by_name, df, tensors, tag):
    report = {}
    for key, net_name, labels, column in HEADS:
        probs = softmax_batches(nets_by_name[net_name], tensors)
        y = np.array([labels.index(str(v)) for v in df[column]])
        pred = probs.argmax(1)
        gold = probs[np.arange(len(y)), y]
        cm = confusion_matrix(y, pred, labels=list(range(len(labels)))).tolist()
        report[key] = {
            "accuracy": float((pred == y).mean()),
            "agree_at_0.5": float((gold >= 0.5).mean()),
            "labels": labels,
            "confusion_recorded_rows_predicted_cols": cm,
        }
        print(f"{tag} {key:6s} acc {report[key]['accuracy']:.3f}  agree@0.5 {report[key]['agree_at_0.5']:.3f}")
        print("  confusion", cm)
    return report


nets = build_concept_nets(DEVICE)
kind_net, shape_net, margin_net, calc_net, backbone = nets
by_name = {"kind_net": kind_net, "shape_net": shape_net, "margin_net": margin_net, "calc_net": calc_net}
model = build_eval_model(nets)
model.load_state(str(STAGE2_PATH))
for net in nets:
    net.eval()

stage2_val_concepts = concept_report(by_name, df_val, val_tensors, "Stage 2 val")
for key, expected in PAPER_AGREE.items():
    got = stage2_val_concepts[key]["agree_at_0.5"]
    if abs(got - expected) > 0.006:
        raise RuntimeError(
            f"Stage 2 validation agree@0.5 for {key} is {got:.3f}, paper is {expected:.3f}. "
            "The images, split, or weights are not the paper run."
        )
print("Stage 2 validation concept check matches the paper.")

saved_test = np.load(TEST_PROB_PATH).astype(float)
if len(saved_test) != len(df_test):
    raise RuntimeError(f"test_prob.npy has {len(saved_test)} rows, test split has {len(df_test)}")
spot = np.linspace(0, len(df_test) - 1, 30).astype(int)
spot_diff = [abs(query_prob(model, "test", int(i)) - float(saved_test[int(i)])) for i in spot]
spot_max = float(max(spot_diff))
print(f"Stage 2 spot check against test_prob.npy: max abs diff {spot_max:.6f} on {len(spot)} images")
if spot_max > 1e-3:
    raise RuntimeError("Stage 2 query probabilities do not match test_prob.npy. Stage 1 would not be comparable.")

load_warmup(nets, WARMUP_PATH)
for net in nets:
    net.eval()
stage1_concepts = {
    "val": concept_report(by_name, df_val, val_tensors, "Stage 1 val"),
    "test": concept_report(by_name, df_test, test_tensors, "Stage 1 test"),
}
y_val, p_val = score_split(model, "val", df_val)
y_test, p_test = score_split(model, "test", df_test)
thr, reached = choose_threshold(y_val, p_val, TARGET_SENSITIVITY)
stage1 = {
    "threshold": thr,
    "reached_target_sensitivity": reached,
    "val_auc": float(roc_auc_score(y_val, p_val)),
    "test_auc": float(roc_auc_score(y_test, p_test)),
    "val_at_0.5": metrics_at(y_val, p_val, 0.5),
    "test_at_0.5": metrics_at(y_test, p_test, 0.5),
    "val_at_threshold": metrics_at(y_val, p_val, thr),
    "test_at_threshold": metrics_at(y_test, p_test, thr),
}
print("Stage 1 malignancy")
print(json.dumps(stage1, indent=2))

model.load_state(str(STAGE2_PATH))
for net in nets:
    net.eval()
stage2_test_concepts = concept_report(by_name, df_test, test_tensors, "Stage 2 test")

payload = {
    "stage1_malignancy": stage1,
    "stage2_spotcheck_max_abs_diff": spot_max,
    "stage1_concepts": stage1_concepts,
    "stage2_concepts": {"val": stage2_val_concepts, "test": stage2_test_concepts},
}
out_path = OUTPUT_DIR / "paper_weight_eval.json"
out_path.write_text(json.dumps(payload, indent=2))
np.save(OUTPUT_DIR / "stage1_val_prob.npy", p_val)
np.save(OUTPUT_DIR / "stage1_test_prob.npy", p_test)
print("Saved", out_path)
print(json.dumps(payload, indent=2))
